# AeroGuard-X — Data & Temporal Model Exploration

This notebook tells the story behind the ML: what the synthetic data looks like,
and *why a temporal model beats a per-frame snapshot*.

> Run cells top-to-bottom (Shift+Enter). Requires the package installed:
> `pip install -e ".[dev,explain]"` from the project root.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from aeroguard.ml.sequence_dataset import generate_sequence_dataset, FRAME_CHANNELS
from aeroguard.ml.temporal import compare_temporal_vs_frame

print("Channels per frame:", FRAME_CHANNELS)

## 1. What does a telemetry window look like?

Each sample is a short *window* of time (15 frames ~ 3 seconds). Impairment is a
**trajectory**, not a snapshot. Let's plot a few windows.

In [ ]:
X, y = generate_sequence_dataset(n_windows=400, seed=0)
print("X shape (windows, frames, channels):", X.shape)
print("Positive (impaired) rate:", y.mean().round(3))

# Plot g-force and perfusion (ppg) over time for one safe and one impaired window
safe_idx = np.where(y == 0)[0][0]
danger_idx = np.where(y == 1)[0][0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, idx, label in [(axes[0], safe_idx, "SAFE"), (axes[1], danger_idx, "IMPAIRED")]:
    ax.plot(X[idx, :, 0], label="g-force", marker="o")
    ax.plot(X[idx, :, 1] * 10, label="ppg x10", marker="s")
    ax.set_title(f"{label} window")
    ax.set_xlabel("frame (time)")
    ax.legend()
plt.tight_layout()
plt.show()

## 2. The key idea: the final frame is deliberately ambiguous

The dangerous (sustained-g) and safe (transient-spike) scenarios are built to
share a similar *final frame*. So a model that only sees the latest instant is
near-blind. Let's prove the temporal model wins on identical held-out data.

In [ ]:
_, report = compare_temporal_vs_frame(n_windows=6000, seed=42)
print(report.summary())

fig, ax = plt.subplots(figsize=(6, 4))
models = ["Per-frame\n(snapshot)", "Temporal\n(trajectory)"]
aucs = [report.frame_roc_auc, report.temporal_roc_auc]
bars = ax.bar(models, aucs, color=["#94a3b8", "#38bdf8"])
ax.set_ylabel("ROC-AUC (held-out)")
ax.set_ylim(0.5, 0.85)
ax.set_title(f"Temporal modelling uplift: +{report.auc_uplift:.3f} AUC")
for b, v in zip(bars, aucs):
    ax.text(b.get_x() + b.get_width()/2, v + 0.005, f"{v:.3f}", ha="center")
plt.tight_layout()
plt.show()

## 3. Is the uplift consistent, or luck?

A single run could be noise. We check across several seeds — the honest way to
claim an improvement.

In [ ]:
uplifts = []
for seed in range(5):
    _, r = compare_temporal_vs_frame(n_windows=6000, seed=seed)
    uplifts.append(r.auc_uplift)
    print(f"seed {seed}: frame={r.frame_roc_auc:.3f}  temporal={r.temporal_roc_auc:.3f}  uplift={r.auc_uplift:+.3f}")

print(f"\nMean uplift: {np.mean(uplifts):+.4f}  |  wins: {sum(u>0 for u in uplifts)}/5")

## Takeaway

The temporal model consistently beats the snapshot baseline by a **modest,
real** margin (~+0.04 AUC), because it can read the *path* to a frame, not just
the frame. A suspiciously huge gap would suggest leakage; this is honest signal.

Next: see `02_explainability.ipynb` for how each advisory is explained with SHAP.